# JSON API 방식 핵심 개념
요청하신 코드에 대한 상세한 분석과 함께, 웹 크롤링과 대비되는 **JSON API 방식의 핵심 개념**을 일목요연하게 정리해 드립니다.

## 1. 소스코드 상세 설명 (Line-by-Line)

- **`SERVICE_KEY = ''`**
- 공공데이터를 이용하기 위해 발급받는 고유 인증키를 넣는 변수
- **`url = f'.../json/tbLnOpendataRtmsV/1/1000/2025'`**
- 서울시 API 호환 규칙에 맞춘 요청 주소입니다. 뒤에 붙은 인자들은 서울시 데이터 규격에 따라 순서대로 `[출력형식(json)] / [서비스명] / [요청시작번호(1)] / [요청끝번호(1000)] / [계약년도(2025)]`를 의미합니다.
  + 즉, 2025년 데이터 중 상위 1,000건을 달라는 요청입니다.
- **`req = requests.get(url)` 및 `content = req.json()**`
  + 지정한 URL로 데이터를 요청하고, 서버가 돌려준 응답을 파이썬이 즉시 다룰 수 있는 자료구조(딕셔너리 및 리스트)로 변환(`파싱`)하는 핵심 구간입니다.


## 2. JSON API 데이터 수집 방식의 이해

- 일반적인 웹 크롤링(스크래핑)과 이번에 사용하신 API 방식은 데이터를 취득하는 목적지와 효율성 면에서 큰 차이가 있습니다.

### 일반 웹 크롤링 (HTML 파싱)

- 사람이 모니터로 보는 웹 화면(HTML)을 통째로 받아온 뒤, 복잡한 태그 사이에서 필요한 글자만 찾아내는 방식
- 웹사이트의 디자인이나 메뉴 위치가 바뀌면 코드가 쉽게 망가지고, 불필요한 이미지나 레이아웃 정보까지 다운로드하므로 비효율적

### JSON API 방식 (구조화된 데이터 통신)

- 서버가 디자인 요소를 모두 배제하고, 프로그램과 프로그램이 오직 "순수한 데이터"만 주고받기로 약속한 전용 통로를 이용하는 방식입니다.
- 화면이 어떻게 바뀌든 데이터의 본질적인 구조는 유지되므로 수집 프로그램이 매우 안정적이며, 처리 속도가 압도적으로 빠릅니다.

---

## 3. JSON (JavaScript Object Notation) 데이터 구조

API 통신에서 전 세계 표준처럼 사용되는 **JSON**은 데이터를 저장하거나 전송할 때 사용하는 텍스트 포맷입니다. 자바스크립트 언어에서 파생되었으나, 구조가 파이썬의 자료구조와 1:1로 매핑되어 다루기가 매우 쉽습니다.

### JSON과 파이썬의 매핑 관계

- **중괄호 `{ }` (JSON Object) $\rightarrow$ 파이썬 딕셔너리(Dictionary)**
- `Key: Value` 쌍으로 매핑된 구조입니다. 코드에서 `content['tbLnOpendataRtmsV']` 형태로 대괄호를 사용하여 접근할 수 있었던 이유가 바로 데이터가 딕셔너리 형태로 파싱되었기 때문입니다.


- **대괄호 `[ ]` (JSON Array) $\rightarrow$ 파이썬 리스트(List)**
- 데이터가 순서대로 나열된 목록입니다. 위의 `row` 키 안에는 각 부동산 거래 건수들이 `[ {1번거래}, {2번거래}, {3번거래} ... ]` 처럼 딕셔너리를 품은 리스트 구조로 들어있습니다. Pandas는 이 구조를 가장 깔끔하게 표(DataFrame)로 변환해 줍니다.



---

## 주요 Key Point

- **페이징(Paging) 처리의 필요성**: 서울시 오픈 API는 서버 부하를 막기 위해 보통 1회 요청 시 최대 1,000건의 데이터만 반환하도록 제한을 걸어둡니다. 만약 2025년 전체 데이터를 수집하려면 반복문(`for` 또는 `while`)을 사용하여 URL의 번호 구간을 `1/1000`, `1001/2000`, `2001/3000` 형태로 동적으로 바꾸며 누적 적재하는 코드로 확장해야 합니다.



In [1]:
import requests  # 웹 서버와 HTTP 통신을 하기 위한 라이브러리 임포트
import json      # JSON 형식의 데이터를 다루기 위한 표준 라이브러리 임포트
import pandas as pd  # 수집한 데이터를 표(DataFrame) 구조로 가공하기 위한 라이브러리 임포트

# 서울시 열린데이터광장에서 발급받은 인증키 설정 (개인 고유 식별 키)
SERVICE_KEY = '526a65735170707038307546447973'

# Open API 요청 전용 URL 생성 (f-string 활용)
# 규칙: [인증키] / [출력형식(json)] / [서비스명(부동산매매일반)] / [시작번호(1)] / [끝번호(1000)] / [계약년도(2025)]
url = f'http://openapi.seoul.go.kr:8088/{SERVICE_KEY}/json/tbLnOpendataRtmsV/1/1000/2025'

# 설정한 API URL로 HTTP GET 요청을 보내고 서버의 응답 객체를 수령
req = requests.get(url)

# 서버가 리턴한 JSON 포맷의 텍스트 데이터를 파이썬 딕셔너리(Dict) 구조로 자동 변환
content = req.json()

# 변환된 전체 데이터 중 최상위 키('tbLnOpendataRtmsV') 내부의 실제 거래 데이터 배열('row')만 추출
# con 변수에는 [ {1번거래정보}, {2번거래정보}, ... ] 형태의 딕셔너리 리스트가 담김
con = content['tbLnOpendataRtmsV']['row']

# 파이썬 리스트 구조의 데이터를 행과 열을 갖춘 Pandas의 2차원 데이터프레임(표)으로 변환
result = pd.DataFrame(con)

# 최종 생성된 데이터프레임의 형상(행의 개수, 열의 개수)을 튜플 형태로 확인 및 출력
# 예: 정상 수집 시 (1000, 변수개수) 형태로 출력됨
result.shape

(1000, 21)

In [6]:
result

,RCPT_YR,CGG_CD,CGG_NM,STDG_CD,STDG_NM,LOTNO_SE,LOTNO_SE_NM,MNO,SNO,BLDG_NM,...,THING_AMT,ARCH_AREA,LAND_AREA,FLR,RGHT_SE,RTRCN_DAY,ARCH_YR,BLDG_USG,DCLR_SE,OPBIZ_RESTAGNT_SGG_NM
0,2025,11350,노원구,10400,하계동,1,대지,0273,0000,장미(시영6),...,71500,59.460,0.0,13.0,,20260316,1989,아파트,중개거래,서울 노원구
1,2025,11230,동대문구,10600,장안동,1,대지,0336,0000,장안현대홈타운(336),...,123000,112.780,0.0,20.0,,,2003,아파트,중개거래,서울 동대문구
2,2025,11350,노원구,10300,공릉동,1,대지,0106,0000,공릉해링턴플레이스,...,86000,99.917,0.0,10.0,,,2000,아파트,중개거래,서울 노원구
3,2025,11350,노원구,10500,상계동,1,대지,1078,0000,현대2차,...,54000,82.680,0.0,10.0,,,1993,아파트,중개거래,서울 노원구
4,2025,11620,관악구,10200,신림동,1,대지,1735,0000,관악산휴먼시아2단지,...,70000,84.980,0.0,9.0,,,2008,아파트,중개거래,서울 관악구
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,2025,11380,은평구,10200,녹번동,1,대지,0281,0000,북한산푸르지오,...,108500,84.990,0.0,12.0,,,2015,아파트,중개거래,서울 은평구
996,2025,11560,영등포구,10700,영등포동6가,1,대지,0042,0000,에이클래스,...,30200,50.030,61.5,2.0,,20260102,null,오피스텔,중개거래,경남 창원시 성산구
997,2025,11740,강동구,10900,천호동,1,대지,0026,0014,명성하이츠빌라,...,27000,51.360,23.0,3.0,,,1991,연립다세대,중개거래,서울 송파구
998,2025,11710,송파구,10400,송파동,1,대지,0119,0000,한양아파트,...,244000,119.680,0.0,10.0,,20260126,1983,아파트,중개거래,서울 송파구


In [8]:
con

[{'RCPT_YR': '2025',
  'CGG_CD': '11350',
  'CGG_NM': '노원구',
  'STDG_CD': '10400',
  'STDG_NM': '하계동',
  'LOTNO_SE': '1',
  'LOTNO_SE_NM': '대지',
  'MNO': '0273',
  'SNO': '0000',
  'BLDG_NM': '장미(시영6)',
  'CTRT_DAY': '20251231',
  'THING_AMT': '71500',
  'ARCH_AREA': 59.46,
  'LAND_AREA': 0.0,
  'FLR': 13.0,
  'RGHT_SE': '',
  'RTRCN_DAY': '20260316',
  'ARCH_YR': '1989',
  'BLDG_USG': '아파트',
  'DCLR_SE': '중개거래',
  'OPBIZ_RESTAGNT_SGG_NM': '서울 노원구'},
 {'RCPT_YR': '2025',
  'CGG_CD': '11230',
  'CGG_NM': '동대문구',
  'STDG_CD': '10600',
  'STDG_NM': '장안동',
  'LOTNO_SE': '1',
  'LOTNO_SE_NM': '대지',
  'MNO': '0336',
  'SNO': '0000',
  'BLDG_NM': '장안현대홈타운(336)',
  'CTRT_DAY': '20251231',
  'THING_AMT': '123000',
  'ARCH_AREA': 112.78,
  'LAND_AREA': 0.0,
  'FLR': 20.0,
  'RGHT_SE': '',
  'RTRCN_DAY': '',
  'ARCH_YR': '2003',
  'BLDG_USG': '아파트',
  'DCLR_SE': '중개거래',
  'OPBIZ_RESTAGNT_SGG_NM': '서울 동대문구'},
 {'RCPT_YR': '2025',
  'CGG_CD': '11350',
  'CGG_NM': '노원구',
  'STDG_CD': '10300',
  'STD

## 멀티스레드 활용

In [5]:
import time
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

# -------------------------------------------------------------------------
# [기본 설정] 인증키 및 베이스 정보 정의
# -------------------------------------------------------------------------
SERVICE_KEY = '526a65735170707038307546447973'
SERVICE_NAME = 'tbLnOpendataRtmsV'
YEAR = '2025'

# 공통 요청 헤더 (봇 차단 방지용 안전장치)
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# -------------------------------------------------------------------------
# [Phase 1] 동기 처리: 2025년 전체 데이터 개수(Total Count) 파악
# -------------------------------------------------------------------------
print("1단계: 2025년 전체 데이터 건수 조회 중...")
check_url = f'http://openapi.seoul.go.kr:8088/{SERVICE_KEY}/json/{SERVICE_NAME}/1/1/{YEAR}'

res = requests.get(check_url, headers=HEADERS)
if res.status_code != 200:
    print("메인 API 서버 접속 실패")
    exit()

meta_data = res.json()
# 서울시 API가 제공하는 '전체 행 개수' 추출
total_count = meta_data[SERVICE_NAME]['list_total_count']
print(f"▶ [확인 완료] 2025년 총 데이터 건수: {total_count}건")
print("-" * 60)


# -------------------------------------------------------------------------
# [Phase 2] 주소 생성: 1,000건 단위로 분할된 요청 URL 리스트 구축
# -------------------------------------------------------------------------
# 서울시 제한량인 1,000단위로 시작점과 끝점을 계산하여 주소 생성
# 예: 1~1000, 1001~2000, 2001~3000 ...
api_urls = []
for start in range(1, total_count + 1, 1000):
    end = start + 999
    if end > total_count:
        end = total_count

    url = f'http://openapi.seoul.go.kr:8088/{SERVICE_KEY}/json/{SERVICE_NAME}/{start}/{end}/{YEAR}'
    api_urls.append(url)


# -------------------------------------------------------------------------
# [Phase 3] 병렬 작업 정의: 단일 1,000건 덩어리를 가져오는 함수
# -------------------------------------------------------------------------
def fetch_api_chunk(url):
    """지정된 구간(1,000건)의 JSON 데이터를 수집하여 리스트로 반환"""
    try:
        req = requests.get(url, headers=HEADERS, timeout=10) # 10초 타임아웃
        if req.status_code != 200:
            return None

        content = req.json()
        # 해당 구간의 row 데이터 추출
        return content[SERVICE_NAME]['row']
    except Exception as e:
        return None


# -------------------------------------------------------------------------
# [Phase 4] 멀티스레딩 실행: ThreadPoolExecutor 활용 비동기 고속 수집
# -------------------------------------------------------------------------
print("2단계: 멀티스레드 기반 1,000건 단위 동시 병렬 수집 시작...")
all_rows = []
start_time = time.time()

# 동시 일꾼(스레드) 수 설정. 공공데이터 API이므로 서버 과부하를 고려해 5~8개가 안전합니다.
MAX_WORKERS = 5

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # 쪼개놓은 URL들을 스레드 풀에 비동기로 전부 예약 처리
    futures = {executor.submit(fetch_api_chunk, url): url for url in api_urls}

    # 먼저 통신이 끝난 작업 순서대로 데이터를 실시간 취합
    for idx, future in enumerate(as_completed(futures), 1):
        chunk_data = future.result()
        if chunk_data:
            all_rows.extend(chunk_data) # 리스트 내에 딕셔너리들을 누적 적재

        print(f"▶ 병렬 처리 진행률: [{idx}/{len(api_urls)}] 완료 (현재 누적 데이터: {len(all_rows)}건)")

# -------------------------------------------------------------------------
# [Phase 5] 데이터프레임 빌드 및 벤치마크 결과 확인
# -------------------------------------------------------------------------
df_seoul_2025 = pd.DataFrame(all_rows)
end_time = time.time()

print("\n==================== [서울시 API 병렬 수집 완결 리포트] ====================")
print(f"▶ 총 소요 시간 : {end_time - start_time:.2f} 초")
print(f"▶ 목표 데이터  : {total_count} 건")
print(f"▶ 최종 수집량  : {df_seoul_2025.shape[0]} 건")
print(f"▶ 마스터 데이터프레임 형상 (행, 열): {df_seoul_2025.shape}")
print("==========================================================================")

# 최종 데이터 셋 로컬 CSV 저장 (한글 깨짐 방지 utf-8-sig 옵션 적용)
df_seoul_2025.to_csv("seoul_real_estate_2025_master.csv", index=False, encoding="utf-8-sig")
print("▶ 'seoul_real_estate_2025_master.csv' 저장 완료.")

1단계: 2025년 전체 데이터 건수 조회 중...
▶ [확인 완료] 2025년 총 데이터 건수: 134986건
------------------------------------------------------------
2단계: 멀티스레드 기반 1,000건 단위 동시 병렬 수집 시작...
▶ 병렬 처리 진행률: [1/135] 완료 (현재 누적 데이터: 1000건)
▶ 병렬 처리 진행률: [2/135] 완료 (현재 누적 데이터: 2000건)
▶ 병렬 처리 진행률: [3/135] 완료 (현재 누적 데이터: 3000건)
▶ 병렬 처리 진행률: [4/135] 완료 (현재 누적 데이터: 4000건)
▶ 병렬 처리 진행률: [5/135] 완료 (현재 누적 데이터: 5000건)
▶ 병렬 처리 진행률: [6/135] 완료 (현재 누적 데이터: 6000건)
▶ 병렬 처리 진행률: [7/135] 완료 (현재 누적 데이터: 7000건)
▶ 병렬 처리 진행률: [8/135] 완료 (현재 누적 데이터: 8000건)
▶ 병렬 처리 진행률: [9/135] 완료 (현재 누적 데이터: 9000건)
▶ 병렬 처리 진행률: [10/135] 완료 (현재 누적 데이터: 10000건)
▶ 병렬 처리 진행률: [11/135] 완료 (현재 누적 데이터: 11000건)
▶ 병렬 처리 진행률: [12/135] 완료 (현재 누적 데이터: 12000건)
▶ 병렬 처리 진행률: [13/135] 완료 (현재 누적 데이터: 13000건)
▶ 병렬 처리 진행률: [14/135] 완료 (현재 누적 데이터: 14000건)
▶ 병렬 처리 진행률: [15/135] 완료 (현재 누적 데이터: 15000건)
▶ 병렬 처리 진행률: [16/135] 완료 (현재 누적 데이터: 16000건)
▶ 병렬 처리 진행률: [17/135] 완료 (현재 누적 데이터: 17000건)
▶ 병렬 처리 진행률: [18/135] 완료 (현재 누적 데이터: 18000건)
▶ 병렬 처리 진행률: [19/135] 완료 (현재 누적 데이터: 19000건)


# XML 데이터 구조 및 파싱 매커니즘 이해

## 1. XML(Extensible Markup Language)이란?

- XML은 HTML처럼 태그(`< >`)를 사용하여 데이터의 구조와 의미를 정의하는 마크업 언어입니다.

### JSON과의 가장 큰 차이점

- **JSON**은 중괄호`{}`와 대괄호`[]`를 기반으로 한 속성-값 쌍의 데이터 포맷입니다.
- **XML**은 사람이 읽을 수 있는 문서 형식에 가까우며, 사용자가 직접 태그 이름(예: `<row>`, `<Title>`)을 정의하여 데이터의 계층 구조(부모-자식 관계)를 명확하게 표현할 수 있다는 장점이 있습니다. 공공데이터포털이나 오래된 대형 기관의 API에서 여전히 표준으로 많이 사용됩니다.

---

## 2. 메모에 작성된 XML 파싱 프로세스 상세 분석

- 파이썬의 표준 라이브러리인 `xml.etree.ElementTree`(보통 `import xml.etree.ElementTree as ET`로 임포트)를 사용할 때의 데이터 처리 흐름입니다.

### [1] `ET.fromstring(xml_text)` $\rightarrow$ DOM 트리 생성

- 서버로부터 받은 XML 통짜 문자열을 파이썬이 탐색할 수 있는 거대한 나무(Tree) 모양의 객체 구조로 변환합니다. 이 나무의 가장 시작점을 **루트(Root) 노드**라고 부릅니다.

### [2] `root.find(SERVICE)` $\rightarrow$ 메인 진입점 탐색

- XML 내부는 인증 상태 코드, 전체 건수 등 다양한 메타데이터 태그가 섞여 있습니다. `find()` 메서드를 통해 진짜 데이터가 시작되는 메인 서비스 태그(예: `<tbLnOpendataRtmsV>`)의 위치로 단번에 이동합니다.

### [3] `service_node.findall("row")` $\rightarrow$ 데이터 루프

- 데이터의 핵심 알맹이들이 `<row> ... </row>` 태그로 감싸져서 수백 개가 나열되어 있습니다. `findall()`은 일치하는 모든 태그를 리스트 형태로 한 번에 싹 긁어모아 반복문(`for`)을 돌릴 수 있게 해줍니다.

### [4] 자식 태그를 `dict`로 변환 후 `DataFrame` 빌드

```python
# 내부적으로 일어나는 메커니즘 예시
row_dict = {}
for child in row:
    row_dict[child.tag] = child.text  # 태그명은 Key가 되고, 내부 글자는 Value가 됨

```

- XML은 JSON과 달리 파이썬 데이터 타입과 바로 호환되지 않으므로, 각 `<row>` 안의 태그와 텍스트를 위와 같이 **딕셔너리(`{Key: Value}`) 형태로 재가공**해야 합니다.
- 이렇게 딕셔너리들이 쌓인 리스트(`rows`)를 `pd.DataFrame(rows)`에 입력하면 Pandas가 완벽한 표 형태로 변환해 줍니다.

### [5] `to_excel()` 영구 저장

- `to_csv()`가 텍스트 기반 저장 방식이라면, `to_excel("파일명.xlsx", index=False)`은 실제 마이크로소프트 엑셀 규격파일로 깔끔하게 내보내는 메서드입니다. 데이터프레임 구조가 그대로 엑셀 시트로 변환되므로 비전공자나 실무진과 데이터를 공유할 때 가장 유용합니다.

---

## 요약 및 팁

- XML은 **태그 기반의 계층형(Tree) 구조**를 가집니다.
- 파이썬으로 다룰 때는 `ElementTree`를 이용해 **문자열 파싱 $\rightarrow$ 특정 태그 검색 $\rightarrow$ 딕셔너리 변환** 단계를 거쳐 Pandas로 연동하는 것이 정석입니다.


In [2]:
import requests                     # 웹 서버와 HTTP 통신을 하기 위한 라이브러리 임포트
import xml.etree.ElementTree as ET  # XML 데이터를 트리 구조로 파싱하고 탐색하는 표준 라이브러리 임포트
import pandas as pd                 # 수집한 데이터를 표(DataFrame) 형태로 가공하기 위한 라이브러리 임포트

# 서울시 열린데이터광장에서 발급받은 인증키 설정
SERVICE_KEY = '526a65735170707038307546447973'

# Open API 요청 URL 생성 (데이터 포맷을 /xml/로 지정)
# 규칙: [인증키] / [출력형식(xml)] / [서비스명] / [시작번호(1)] / [끝번호(1000)] / [계약년도(2025)]
url = f'http://openapi.seoul.go.kr:8088/{SERVICE_KEY}/xml/tbLnOpendataRtmsV/1/1000/2025'

# 설정한 URL로 HTTP GET 요청을 보내고 응답 객체를 수령
resp = requests.get(url)
print("Status Code:", resp.status_code) # 통신 상태 코드 출력 (200: 성공, 403: 인증실패, 404: 주소오류 등)

# 웹 서버와의 통신이 정상적으로 성공(200)했을 경우에만 내부 로직 실행
if resp.status_code == 200:
    # 응답받은 본문(XML 문자열)의 앞뒤 불필요한 공백 제거 후 변수 저장
    xml_text = resp.text.strip()

    # XML 파싱 시작: 텍스트 형태의 XML을 파이썬이 탐색 가능한 Element 객체(트리 구조)로 변환
    # 최상위 루트(Root) 노드 객체를 반환받음
    root = ET.fromstring(xml_text)

    # 각 행(row) 데이터를 딕셔너리 형태로 가공하여 담을 마스터 리스트 생성
    rows = []

    # root 노드 직속 하위에 존재하는 모든 <row> 태그를 찾아서 반복문(Loop) 수행
    for row in root.findall("row"):
        row_dict = {} # 하나의 <row> 안의 데이터를 담을 임시 딕셔너리 생성

        # <row> 내부의 자식 태그들(예: <ACC_YEAR>, <SGG_NM> 등)을 하나씩 순회
        for child in row:
            # child.tag(태그 이름)를 Key로, child.text(태그 내부의 문자열 값)를 Value로 매핑
            row_dict[child.tag] = child.text

        # 딕셔너리로 변환 완료된 한 건의 거래 데이터 구조를 마스터 리스트(rows)에 추가
        rows.append(row_dict)

    # 변환된 딕셔너리 리스트( [{}, {}, ...] )를 Pandas의 2차원 표(DataFrame) 구조로 전환
    df = pd.DataFrame(rows)

    # 최종 생성된 데이터프레임의 구조(행의 개수, 열의 개수) 확인 및 출력
    print("DataFrame shape:", df.shape)
    # 데이터프레임의 상위 5개 행 데이터를 콘솔에 미리보기 형태로 출력
    print(df.head())

else:
    # HTTP 통신 에러 발생 시, 원인 파악을 위해 에러 메세지 본문 앞부분 300자 출력
    print("HTTP 에러:", resp.text[:300])

Status Code: 200
DataFrame shape: (1000, 21)
  RCPT_YR CGG_CD CGG_NM STDG_CD STDG_NM LOTNO_SE LOTNO_SE_NM   MNO   SNO  \
0    2025  11230   동대문구   10800     회기동        1          대지  0065  0000   
1    2025  11560   영등포구   13200     신길동        1          대지  4967  0000   
2    2025  11590    동작구   10900    신대방동        1          대지  0710  0000   
3    2025  11560   영등포구   13200     신길동        1          대지  4964  0000   
4    2025  11350    노원구   10400     하계동        1          대지  0271  0003   

      BLDG_NM  ... THING_AMT ARCH_AREA LAND_AREA FLR RGHT_SE RTRCN_DAY  \
0         신현대  ...     69000     60.48  0.000000   1    None      None   
1      신길파크자이  ...    170000     84.99  0.000000   1    None      None   
2        경남교수  ...    127500    113.93  0.000000   6    None      None   
3  신길센트럴자이아파트  ...    167000     59.98  0.000000  24    None      None   
4          건영  ...     42000     45.55  0.000000   3    None      None   

  ARCH_YR BLDG_USG DCLR_SE OPBIZ_RESTAGNT_SGG_NM  
0 

In [3]:
df.shape

(1000, 21)

In [7]:
df

,RCPT_YR,CGG_CD,CGG_NM,STDG_CD,STDG_NM,LOTNO_SE,LOTNO_SE_NM,MNO,SNO,BLDG_NM,...,THING_AMT,ARCH_AREA,LAND_AREA,FLR,RGHT_SE,RTRCN_DAY,ARCH_YR,BLDG_USG,DCLR_SE,OPBIZ_RESTAGNT_SGG_NM
0,2025,11230,동대문구,10800,회기동,1,대지,0065,0000,신현대,...,69000,60.48,0.000000,1,None,None,1989,아파트,중개거래,서울 동대문구
1,2025,11560,영등포구,13200,신길동,1,대지,4967,0000,신길파크자이,...,170000,84.99,0.000000,1,None,None,2022,아파트,중개거래,서울 영등포구
2,2025,11590,동작구,10900,신대방동,1,대지,0710,0000,경남교수,...,127500,113.93,0.000000,6,None,None,2001,아파트,중개거래,서울 동작구
3,2025,11560,영등포구,13200,신길동,1,대지,4964,0000,신길센트럴자이아파트,...,167000,59.98,0.000000,24,None,None,2021,아파트,중개거래,서울 영등포구
4,2025,11350,노원구,10400,하계동,1,대지,0271,0003,건영,...,42000,45.55,0.000000,3,None,None,1988,아파트,중개거래,서울 노원구
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,2025,11650,서초구,10800,서초동,1,대지,1630,0001,서초레이나2,...,44006,23.8,14.000000,3,None,None,2024,연립다세대,직거래,None
996,2025,11650,서초구,10800,서초동,1,대지,1630,0006,서초레이나1,...,43990,23.7,14.000000,3,None,None,2024,연립다세대,직거래,None
997,2025,11650,서초구,10800,서초동,1,대지,1630,0001,서초레이나2,...,44006,23.8,14.000000,5,None,None,2024,연립다세대,직거래,None
998,2025,11560,영등포구,12700,양평동3가,1,대지,0078,0020,포엠빌,...,32514,29.23,32.060000,11,None,None,2023,오피스텔,중개거래,서울 마포구


## 2025년 전체 데이터 가져오기

In [4]:
import time
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm  # 진행 상황을 시각적인 바(Bar)로 출력하기 위한 라이브러리

# -------------------------------------------------------------------------
# [기본 설정] 인증키 및 베이스 정보 정의
# -------------------------------------------------------------------------
SERVICE_KEY = '526a65735170707038307546447973'
SERVICE_NAME = 'tbLnOpendataRtmsV'
YEAR = '2025'
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# -------------------------------------------------------------------------
# [Phase 1] 전체 데이터 개수 파악 (XML 포맷 호출)
# -------------------------------------------------------------------------
print("1단계: XML API를 통한 2025년 전체 데이터 건수 조회 중...")
check_url = f'http://openapi.seoul.go.kr:8088/{SERVICE_KEY}/xml/{SERVICE_NAME}/1/1/{YEAR}'

res = requests.get(check_url, headers=HEADERS)
if res.status_code != 200:
    print("메인 API 서버 접속 실패")
    exit()

# XML 문자열 파싱하여 list_total_count 태그 검색
root = ET.fromstring(res.text.strip())
total_count = int(root.find("list_total_count").text)
print(f"▶ [확인 완료] 2025년 총 데이터 건수: {total_count}건")
print("-" * 70)


# -------------------------------------------------------------------------
# [Phase 2] 1,000건 단위 분할 XML 요청 URL 리스트 생성
# -------------------------------------------------------------------------
api_urls = []
for start in range(1, total_count + 1, 1000):
    end = start + 999
    if end > total_count:
        end = total_count

    url = f'http://openapi.seoul.go.kr:8088/{SERVICE_KEY}/xml/{SERVICE_NAME}/{start}/{end}/{YEAR}'
    api_urls.append(url)


# -------------------------------------------------------------------------
# [Phase 3] 병렬 작업 정의: 단일 1,000건 XML 뭉치를 가공하는 함수
# -------------------------------------------------------------------------
def fetch_xml_chunk(url):
    """지정된 구간의 XML 데이터를 요청하여 파이썬 딕셔너리 리스트로 변환"""
    try:
        req = requests.get(url, headers=HEADERS, timeout=15)
        if req.status_code != 200:
            return None

        chunk_root = ET.fromstring(req.text.strip())
        chunk_rows = []

        # <row> 태그 내부 요소들을 딕셔너리로 맵핑
        for row in chunk_root.findall("row"):
            row_dict = {}
            for child in row:
                row_dict[child.tag] = child.text
            chunk_rows.append(row_dict)

        return chunk_rows
    except Exception as e:
        return None


# -------------------------------------------------------------------------
# [Phase 4] ThreadPoolExecutor + tqdm 결합 고속 병렬 수집
# -------------------------------------------------------------------------
print("2단계: 멀티스레드 기반 XML 데이터 고속 병렬 수집 시작 (tqdm 활성화)...")
all_rows = []

# [전체 시간 측정 시작]
start_time = time.time()

# 공공 데이터 서버 안정성을 고려한 최적의 스레드 수 (일꾼 5명)
MAX_WORKERS = 5

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # 쪼개놓은 XML 주소들을 스레드 풀에 비동기로 예약
    futures = {executor.submit(fetch_xml_chunk, url): url for url in api_urls}

    # tqdm으로 전체 작업 수(total)를 지정하여 실시간 진행바를 콘솔에 출력
    # as_completed를 감싸서 작업이 끝날 때마다 바가 업데이트되도록 유도
    for future in tqdm(as_completed(futures), total=len(api_urls), desc="서울시 XML 데이터 수집률"):
        chunk_data = future.result()
        if chunk_data:
            all_rows.extend(chunk_data) # 1,000건의 데이터 리스트를 마스터 리스트에 병합

# [전체 시간 측정 종료]
end_time = time.time()
total_execution_time = end_time - start_time


# -------------------------------------------------------------------------
# [Phase 5] 최종 데이터프레임 빌드 및 결과 보고서 출력
# -------------------------------------------------------------------------
df_seoul_xml_2025 = pd.DataFrame(all_rows)

print("\n==================== [서울시 XML API 병렬 수집 완결 리포트] ====================")
print(f"▶ 총 소요 시간 : {total_execution_time:.2f} 초 (약 {total_execution_time/60:.1f} 분)")
print(f"▶ 목표 데이터  : {total_count} 건")
print(f"▶ 최종 수집량  : {df_seoul_xml_2025.shape[0]} 건")
print(f"▶ 마스터 데이터프레임 형상 (행, 열): {df_seoul_xml_2025.shape}")
print("==========================================================================")

# 최종 전체 데이터 셋 엑셀 파일 형태로 저장
df_seoul_xml_2025.to_excel("seoul_real_estate_2025_xml_master.xlsx", index=False)
print("▶ 'seoul_real_estate_2025_xml_master.xlsx' 저장 완료.")

1단계: XML API를 통한 2025년 전체 데이터 건수 조회 중...
▶ [확인 완료] 2025년 총 데이터 건수: 134986건
----------------------------------------------------------------------
2단계: 멀티스레드 기반 XML 데이터 고속 병렬 수집 시작 (tqdm 활성화)...


서울시 XML 데이터 수집률: 100%|██████████| 135/135 [01:33<00:00,  1.45it/s]



==================== [서울시 XML API 병렬 수집 완결 리포트] ====================
▶ 총 소요 시간 : 93.21 초 (약 1.6 분)
▶ 목표 데이터  : 134986 건
▶ 최종 수집량  : 97986 건
▶ 마스터 데이터프레임 형상 (행, 열): (97986, 21)
▶ 'seoul_real_estate_2025_xml_master.xlsx' 저장 완료.
